# Ask Alfonso · training on a GPU (v5)

v5 trains a **bigger model (6 layers, 384-dim, ~12M parameters)** that first reads ~50M
tokens of Wikipedia (WikiText-103) to learn English, then fine-tunes on Alfonso's Q&A
**twice, with two learning rates**. Cell 11 compares both against v4 on the same dev set.

**Before you start:** `Runtime → Change runtime type → T4 GPU → Save`, then `Runtime → Run all`.
Cell 2 asks for permission to use your Google Drive: every stage saves its progress there.

**It takes several hours.** If Colab disconnects: reconnect, `Run all` again. Every stage
continues where it stopped (pretraining every 500 steps, fine-tuning every epoch), and stages
that already finished are skipped. If the output stops moving but the stop button still spins,
it's usually still running: check the `.log` files in `MyDrive/ask-alfonso/checkpoints/`.


In [ ]:
# 1. Check the GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
print("PyTorch", torch.__version__, "· CUDA available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU: Runtime → Change runtime type → T4 GPU"

In [ ]:
# 2. Settings, and Google Drive for checkpoints (survives disconnects)
MODEL = "--n-layer 6 --n-head 6 --n-embd 384"     # ~12M parameters (v1–v4: 4 layers, 256-dim, ~4M)
PRETRAIN_RUN = "v5-pretrain"                       # stage A, shared by both fine-tuning runs
RUNS = {                                           # stage B: same start, two learning rates
    "colab-v5a": "--lr 5e-4 --min-lr 5e-5",        # gentler: keeps more of what pretraining learned
    "colab-v5b": "--lr 1e-3 --min-lr 1e-4",        # v1–v4's learning rate
}

import os
from google.colab import drive
drive.mount("/content/drive")
CKPT = "/content/drive/MyDrive/ask-alfonso/checkpoints"
os.makedirs(CKPT, exist_ok=True)
print("Checkpoints and logs →", CKPT)


In [ ]:
# 3. Get the code (re-running this cell pulls the latest version)
import os
REPO = "https://github.com/Alfonso3435/portfolio.git"
if not os.path.exists("/content/portfolio"):
    !git clone -q {REPO} /content/portfolio
%cd /content/portfolio
!git pull -q
!git log -1 --format="Commit %h · %s"

In [ ]:
# 4. Phase 1 — export the site data and build the dataset (needs Node.js for the TypeScript export)
!node --version || (curl -fsSL https://deb.nodesource.com/setup_20.x | bash - > /dev/null && apt-get install -y -qq nodejs)
!npm ci --ignore-scripts --no-audit --no-fund --loglevel=error
!npm run --silent export:knowledge
!python ml/build_dataset.py --samples 0

In [ ]:
# 5. Phase 2 — train the tokenizer (deterministic: identical to the one on your laptop)
!python ml/train_tokenizer.py | tail -8

### Stage A — learning English from Wikipedia

v4 pretrained on WikiText-2 (~2M words) for 3,000 steps. Its remaining errors were about
*meaning*: "What need does X address?" wasn't recognized as "What problem does X solve?".
Word meanings come from reading a lot of text, so v5 reads ~25× more (the first ~120 MB of
WikiText-103) for 20,000 steps, with a model 3× bigger to hold it. The text and its tokens
are cached in Drive, so a restart doesn't download or encode them again.


In [ ]:
# 6. Download ~120 MB of WikiText-103 (once; cached in Drive)
WIKI = f"{CKPT}/wikitext103-120MB.txt"
if not os.path.exists(WIKI):
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "datasets"], check=True)
    from datasets import load_dataset
    wiki = load_dataset("Salesforce/wikitext", "wikitext-103-raw-v1", split="train")
    size = 0
    with open(WIKI + ".tmp", "w", encoding="utf-8") as f:
        for t in wiki["text"]:
            if t.strip():
                f.write(t)
                size += len(t)
                if size > 120_000_000:
                    break
    os.replace(WIKI + ".tmp", WIKI)
print(f"{os.path.getsize(WIKI) / 1e6:.0f} MB of pretraining text ready")


In [ ]:
# 7. Stage A — pretraining (resumes every 500 steps; skipped once pretrained.pt exists)
!python -u ml/train.py {MODEL} --run {PRETRAIN_RUN} --checkpoints {CKPT} --resume --pretrain-only --pretrain-text {WIKI} --pretrain-steps 20000 --pretrain-lr 6e-4 --pretrain-cache {CKPT}/{PRETRAIN_RUN}/tokens.pt 2>&1 | tee -a {CKPT}/{PRETRAIN_RUN}.log


In [ ]:
# 8. Stage B — fine-tune both runs from the same pretrained weights (each resumes per epoch).
#    Every 4 epochs: dev EM (exactly right), F1 (word overlap), facts (all names/numbers present).
for run, opts in RUNS.items():
    print(f"\n══════ {run} ══════")
    !python -u ml/train.py {MODEL} {opts} --run {run} --checkpoints {CKPT} --resume --init-from {CKPT}/{PRETRAIN_RUN}/pretrained.pt 2>&1 | tee -a {CKPT}/{run}.log


In [ ]:
# 9. Curves — one PNG per run, for the Lab page
import json
import matplotlib.pyplot as plt

for run in RUNS:
    rows = [json.loads(l) for l in open(f"{CKPT}/{run}/metrics.jsonl")]
    ft = [r for r in rows if r["stage"] == "finetune"]
    fig, (a, b) = plt.subplots(1, 2, figsize=(11, 4))
    a.plot([r["epoch"] for r in ft], [r["train_loss"] for r in ft], label="train")
    a.plot([r["epoch"] for r in ft], [r["dev_loss"] for r in ft], label="dev")
    a.set(title=f"{run} · loss (answer tokens)", xlabel="epoch", ylabel="cross-entropy"); a.legend()
    ev = [r for r in ft if "dev_em" in r]
    for key, label in [("dev_em", "exact match"), ("dev_f1", "token F1"), ("dev_facts", "key facts")]:
        b.plot([r["epoch"] for r in ev], [100 * r.get(key, 0) for r in ev], marker="o", label=label)
    b.set(title=f"{run} · dev answers", xlabel="epoch", ylabel="%", ylim=(0, 100)); b.legend()
    plt.tight_layout(); plt.savefig(f"{CKPT}/{run}/curves.png", dpi=150); plt.show()

rows = [json.loads(l) for l in open(f"{CKPT}/{PRETRAIN_RUN}/metrics.jsonl")]
pre = [r for r in rows if r["stage"] == "pretrain"]
plt.figure(figsize=(6, 3.5)); plt.plot([r["step"] for r in pre], [r["train_loss"] for r in pre])
plt.title("Stage A · Wikipedia next-token loss"); plt.xlabel("step"); plt.ylabel("cross-entropy")
plt.tight_layout(); plt.savefig(f"{CKPT}/{PRETRAIN_RUN}/pretrain.png", dpi=150); plt.show()


In [ ]:
# 10. Full pipeline on dev for v4 (baseline) and both v5 runs — same code, same questions.
for run in ["colab-v4", *RUNS]:
    if not os.path.exists(f"{CKPT}/{run}/best.pt"):
        print(f"(no {run}/best.pt, skipped)"); continue
    print(f"\n══════ {run} ══════")
    !python ml/eval_pipeline.py --checkpoint {CKPT}/{run}/best.pt --device cuda --thresholds 0.10 --out {CKPT}/{run}/pipeline_dev.jsonl | tail -7


In [ ]:
# 11. Ask both v5 models the same questions
for run in RUNS:
    print(f"\n══════ {run} ══════")
    !python ml/ask.py --checkpoint {CKPT}/{run}/best.pt --device cuda "What did he build at HackMX?" "Did SkySAR win anything?" "Can you tell me about ZapTap?" "What need does Answr address?" "Does he know Python?" "Has he used React Native?" "What's the capital of France?"


In [ ]:
# 12. Download everything small (model weights, metrics, curves, evaluations, logs).
#     Big files needed only to resume (last.pt, pretrained.pt, tokens.pt) stay in Drive.
names = " ".join([PRETRAIN_RUN, *RUNS])
logs = " ".join(f"{r}.log" for r in [PRETRAIN_RUN, *RUNS])
!cd {CKPT} && rm -f /content/colab-v5.zip && zip -qr /content/colab-v5.zip {names} {logs} -x "*/last.pt" "*/pretrained.pt" "*/tokens.pt" "*.tmp"
from google.colab import files
files.download("/content/colab-v5.zip")
